# 04 — Model Building

We vectorize the `tags` column with `CountVectorizer`, compute pairwise cosine similarity across all
4,799 movies, and wrap it in a `recommend()` function. The fitted movie table and similarity matrix are
pickled to `models/` for the Streamlit app to load directly (no retraining needed at runtime).


In [1]:
import pandas as pd
import numpy as np
import pickle
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("../data/final_movies.csv")
for col in ["genres", "keywords", "cast"]:
    df[col] = df[col].apply(ast.literal_eval)

df.shape


(4799, 13)

## 1. Vectorize tags

In [2]:
cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(df["tags"]).toarray()
print("Vector matrix shape:", vectors.shape)
print("Vocabulary sample:", list(cv.vocabulary_.keys())[:15])


Vector matrix shape: (4799, 5000)
Vocabulary sample: ['centuri', 'marin', 'dispatch', 'moon', 'pandora', 'uniqu', 'mission', 'becom', 'torn', 'follow', 'order', 'protect', 'alien', 'civil', 'action']


## 2. Cosine similarity matrix

In [3]:
similarity = cosine_similarity(vectors)
print("Similarity matrix shape:", similarity.shape)
print(f"Matrix size in memory: {similarity.nbytes / (1024**2):.1f} MB")


Similarity matrix shape: (4799, 4799)
Matrix size in memory: 175.7 MB


## 3. The `recommend()` function

In [4]:
def recommend(movie_title: str, top_n: int = 10):
    matches = df[df["title"].str.lower() == movie_title.lower()]
    if matches.empty:
        return f"'{movie_title}' not found in the dataset."

    idx = matches.index[0]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n + 1]

    results = []
    for i, score in scores:
        results.append({
            "title": df.loc[i, "title"],
            "similarity_score": round(float(score) * 100, 1),
            "genres": df.loc[i, "genres"],
            "vote_average": df.loc[i, "vote_average"],
            "release_year": df.loc[i, "release_year"],
        })
    return pd.DataFrame(results)


## 4. Example recommendations

In [5]:
recommend("Avatar")


,title,similarity_score,genres,vote_average,release_year
0,Aliens,26.6,"[Horror, Action, Thriller, Science Fiction]",7.7,1986
1,Falcon Rising,25.7,"[Adventure, Action]",5.5,2014
2,Titan A.E.,24.7,"[Animation, Action, Science Fiction, Family, A...",6.3,2000
3,Independence Day,24.4,"[Action, Adventure, Science Fiction]",6.7,1996
4,Aliens vs Predator: Requiem,24.3,"[Fantasy, Action, Science Fiction, Thriller, H...",4.9,2007
5,Battle: Los Angeles,24.0,"[Action, Science Fiction]",5.5,2011
6,Small Soldiers,23.4,"[Comedy, Adventure, Fantasy, Science Fiction, ...",6.2,1998
7,Predators,23.4,"[Action, Science Fiction, Adventure, Thriller]",6.0,2010
8,Meet Dave,23.1,"[Comedy, Science Fiction, Adventure, Family]",5.0,2008
9,Ender's Game,23.0,"[Science Fiction, Action, Adventure]",6.6,2013


In [6]:
recommend("The Dark Knight Rises")


,title,similarity_score,genres,vote_average,release_year
0,The Dark Knight,42.6,"[Drama, Action, Crime, Thriller]",8.2,2008
1,Batman Begins,36.1,"[Action, Crime, Drama]",7.5,2005
2,Batman,32.5,"[Fantasy, Action]",7.0,1989
3,Batman Returns,31.6,"[Action, Fantasy]",6.6,1992
4,Batman Forever,30.7,"[Action, Crime, Fantasy]",5.2,1995
5,Batman & Robin,29.7,"[Action, Crime, Fantasy]",4.2,1997
6,Slow Burn,26.6,"[Mystery, Crime, Drama, Thriller]",5.5,2005
7,Nighthawks,25.1,"[Action, Crime, Thriller]",6.4,1981
8,Dead Man Down,24.1,"[Thriller, Action, Crime, Drama]",5.9,2013
9,Defendor,23.6,"[Drama, Action, Comedy, Crime]",6.5,2009


In [7]:
recommend("The Avengers")


,title,similarity_score,genres,vote_average,release_year
0,Avengers: Age of Ultron,35.0,"[Action, Adventure, Science Fiction]",7.3,2015
1,Iron Man 3,30.3,"[Action, Adventure, Science Fiction]",6.8,2013
2,Captain America: Civil War,29.6,"[Adventure, Action, Science Fiction]",7.1,2016
3,Captain America: The First Avenger,28.3,"[Action, Adventure, Science Fiction]",6.6,2011
4,Iron Man 2,28.0,"[Adventure, Action, Science Fiction]",6.6,2010
5,Ant-Man,27.2,"[Science Fiction, Action, Adventure]",7.0,2015
6,Iron Man,26.8,"[Action, Science Fiction, Adventure]",7.4,2008
7,Captain America: The Winter Soldier,25.5,"[Action, Adventure, Science Fiction]",7.6,2014
8,X-Men: First Class,24.4,"[Action, Science Fiction, Adventure]",7.1,2011
9,Thor: The Dark World,24.0,"[Action, Adventure, Fantasy]",6.8,2013


## 5. Evaluate

There's no ground-truth label for "correct" recommendations, so we use a simple proxy metric:
**genre overlap** — what fraction of a query movie's genres appear among its top-10 recommendations'
genres. A content-based model built mostly from genres/keywords/cast/director should score this highly
by construction; it's a sanity check, not a claim of recommendation quality in the human-preference sense.

In [8]:
def genre_overlap_score(movie_title: str, top_n: int = 10) -> float:
    matches = df[df["title"].str.lower() == movie_title.lower()]
    if matches.empty:
        return np.nan
    idx = matches.index[0]
    query_genres = set(df.loc[idx, "genres"])
    if not query_genres:
        return np.nan

    scores = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)[1:top_n + 1]
    overlaps = []
    for i, _ in scores:
        rec_genres = set(df.loc[i, "genres"])
        overlaps.append(len(query_genres & rec_genres) / len(query_genres))
    return float(np.mean(overlaps))


sample_titles = df["title"].sample(40, random_state=42).tolist()
avg_overlap = np.mean([genre_overlap_score(t) for t in sample_titles])
print(f"Average genre-overlap score across 40 random movies: {avg_overlap:.2%}")


Average genre-overlap score across 40 random movies: 70.05%


## 6. Serialize artifacts for the Streamlit app

In [9]:
app_table = df[[
    "movie_id", "title", "overview", "genres", "keywords", "cast",
    "director", "release_year", "vote_average", "vote_count", "popularity", "runtime",
]].copy()

with open("../models/movies.pkl", "wb") as f:
    pickle.dump(app_table, f)

similarity_compact = similarity.astype(np.float16)
with open("../models/similarity.pkl", "wb") as f:
    pickle.dump(similarity_compact, f)

print("Saved models/movies.pkl and models/similarity.pkl")
print("movies.pkl rows:", len(app_table))
print("similarity.pkl shape:", similarity_compact.shape, "| dtype:", similarity_compact.dtype)
print(f"similarity.pkl size on disk will be roughly {similarity_compact.nbytes / (1024**2):.1f} MB")


Saved models/movies.pkl and models/similarity.pkl
movies.pkl rows: 4799
similarity.pkl shape: (4799, 4799) | dtype: float16
similarity.pkl size on disk will be roughly 43.9 MB
